# Bronze Layer — NSW Air Quality Ingestion

Downloads hourly observations from the NSW Air Quality API and lands them
unchanged as JSON files, then reads those files into a Delta table.

Scope: 24 Sydney monitoring stations, 4 pollutants, 2020-2025.

Design rules:
- Raw JSON written to a Volume exactly as received. No cleaning at this stage.
- If downstream logic changes, tables are rebuilt from these files. The API is
  never re-queried.
- Every chunk recorded in a manifest table so nothing can be silently lost.

In [0]:
import os, json, time, requests
from datetime import datetime, UTC
from pyspark.sql import functions as F

BASE = "https://data.airquality.nsw.gov.au/api/Data"
RAW  = "/Volumes/workspace/aq_bronze/raw"
os.makedirs(RAW, exist_ok=True)

POLLUTANTS = ["PM2.5", "PM10", "NO2", "OZONE"]
YEARS      = [2020, 2021, 2022, 2023, 2024, 2025]

RAW is a Volume — governed file storage inside Unity Catalog. Files here survive session restarts, unlike Python variables.

Why the station filter exists

The API returns 137 NSW sites, 26 of them tagged as a Sydney region. Two of those 26 are not stations at all:

Site_Id: 120000000, SiteName: "Sydney north-west", Latitude: None, Longitude: None
Site_Id: 130000000, SiteName: "Sydney south-west", Latitude: None, Longitude: None

A physical monitoring station has coordinates. These are regional aggregates — pre-averaged summaries the API publishes alongside real stations. Include them and every regional average counts that region twice.

The honest filter is on null coordinates, not on the ID being suspiciously large.

## 1. Station selection

Two of the 26 Sydney-region entries are regional aggregates, not physical
stations. Identified by null coordinates and a SiteName identical to the Region.
Excluded to prevent double-counting in regional averages.

In [0]:
sites = requests.get(f"{BASE}/get_SiteDetails").json()

stations = [s for s in sites
            if s["Region"].lower().startswith("sydney")
            and s["Latitude"] is not None
            and s["Longitude"] is not None]

print(f"{len(sites)} NSW sites -> {len(stations)} physical Sydney stations")

137 NSW sites -> 24 physical Sydney stations


Why the chunk is one station, one pollutant, one year

You tested this rather than guessing:

Request	Result
6 pollutants × 1 station × 1 year	HTTP 502 — the upstream server crashed
1 pollutant × 1 station × 1 year	8,760 records, fine
1 pollutant × 1 station × 3 months	2,160 records, fine

So the ceiling sits between those, and one pollutant-year is comfortably under it.

Why EndDate is the following year

Every test came back exactly 24 records short, no matter the window length:

Asked for	Expected	Got
Jan 2024	744	720
Jan–Mar 2024	2,184	2,160
All 2024	8,784	8,760

A constant shortfall of 24 — one day — means the end date is exclusive, not truncation. Confirmed by printing the returned date range: asking for 1–31 January returned 1–30 January.

Left uncorrected, that would have quietly lost one day per station-pollutant-year: 1,296 missing days across the dataset, invisible unless someone counted.

Cell 5 — %md
## 2. Download function

Chunk size is one station, one pollutant, one year. Larger requests cause the
upstream server to return HTTP 502.

The API treats EndDate as EXCLUSIVE: a request ending 2024-01-31 returns data
through 2024-01-30. Requests therefore use the first day of the following year.

Three retries with exponential backoff handle transient upstream failures.

## 2. Download function

Chunk size is one station, one pollutant, one year. Larger requests cause the
upstream server to return HTTP 502.

The API treats EndDate as EXCLUSIVE: a request ending 2024-01-31 returns data
through 2024-01-30. Requests therefore use the first day of the following year.

Three retries with exponential backoff handle transient upstream failures.

In [0]:
def land_chunk(site_id, pollutant, year, retries=3):
    """Download one station-pollutant-year and save the raw JSON, unchanged."""
    body = {
        "Parameters": [pollutant],
        "Sites": [site_id],
        "StartDate": f"{year}-01-01",
        "EndDate":   f"{year + 1}-01-01",   # exclusive
        "Categories": ["Averages"],
        "SubCategories": ["Hourly"],
        "Frequency": ["Hourly average"],
    }

    safe = pollutant.replace(".", "_")
    path = f"{RAW}/site{site_id}_{safe}_{year}.json"

    for attempt in range(retries):
        try:
            r = requests.post(f"{BASE}/get_Observations", json=body, timeout=180)
            if r.status_code == 200:
                records = r.json()
                with open(path, "w") as f:
                    json.dump(records, f)
                return {"site_id": site_id, "pollutant": pollutant, "year": year,
                        "path": path, "records": len(records),
                        "status": "ok", "error": None,
                        "fetched_at": datetime.now(UTC).isoformat()}
            err = f"HTTP {r.status_code}: {r.text[:150]}"
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
        time.sleep(2 ** attempt)

    return {"site_id": site_id, "pollutant": pollutant, "year": year,
            "path": path, "records": 0, "status": "failed", "error": err,
            "fetched_at": datetime.now(UTC).isoformat()}

In [0]:
print(land_chunk(33, "PM2.5", 2024))

## 3. Backfill

576 chunks: 24 stations x 4 pollutants x 6 years. Roughly two hours.

Chunks returning zero records are not failures. They indicate a station that does
not monitor that pollutant, or was not operating that year.

In [0]:
###ite_ids = [s["Site_Id"] for s in stations]
##jobs = [(s, p, y) for s in site_ids for p in POLLUTANTS for y in YEARS]
##print(f"{len(jobs)} chunks queued")

##results = []
#for i, (site, poll, year) in enumerate(jobs, 1):
    #res = land_chunk(site, poll, year)
    #results.append(res)
    #if i % 25 == 0 or res["status"] == "failed":
        #ok = sum(1 for r in results if r["status"] == "ok")
        #print(f"[{i}/{len(jobs)}] ok={ok} last={site}/{poll}/{year} "
#              f"{res['status']} {res['records']}")

#print("FINISHED  ok:", sum(1 for r in results if r["status"] == "ok"),
 #     " failed:", sum(1 for r in results if r["status"] == "failed"))


576 chunks queued
[25/576] ok=25 last=39/PM2.5/2020 ok 8784
[50/576] ok=50 last=70/PM2.5/2021 ok 0
[75/576] ok=75 last=107/PM2.5/2022 ok 8760
[100/576] ok=100 last=171/PM2.5/2023 ok 8760
[125/576] ok=125 last=190/PM2.5/2024 ok 0
[150/576] ok=150 last=206/PM2.5/2025 ok 8760
[175/576] ok=175 last=573/PM10/2020 ok 8784
[200/576] ok=200 last=574/PM10/2021 ok 0
[225/576] ok=225 last=760/PM10/2022 ok 8760
[250/576] ok=250 last=765/PM10/2023 ok 0
[275/576] ok=275 last=919/PM10/2024 ok 8784
[300/576] ok=300 last=1560/PM10/2025 ok 0
[325/576] ok=325 last=1570/NO2/2020 ok 8784
[350/576] ok=350 last=2560/NO2/2021 ok 8760
[375/576] ok=375 last=2570/NO2/2022 ok 8760
[400/576] ok=400 last=113/NO2/2023 ok 8760
[425/576] ok=425 last=155/NO2/2024 ok 8784
[450/576] ok=450 last=1001/NO2/2025 ok 8760
[475/576] ok=475 last=1148/OZONE/2020 ok 8784
[500/576] ok=500 last=1141/OZONE/2021 ok 8760
[525/576] ok=525 last=1750/OZONE/2022 ok 8760
[550/576] ok=550 last=15/OZONE/2023 ok 8760
[575/576] ok=575 last=1007

## 4. Manifest

Built by reading the landed files rather than trusting the loop's in-memory
results. Survives a session restart and records what is actually on disk.

In [0]:
def parse_name(fname):
    """site33_PM2_5_2024.json -> (33, 'PM2.5', 2024)"""
    stem  = fname.replace(".json", "")
    parts = stem.split("_")
    site  = int(parts[0].replace("site", ""))
    year  = int(parts[-1])
    pollutant = "_".join(parts[1:-1]).replace("_", ".")
    return site, pollutant, year

files = sorted(os.listdir(RAW))
manifest = []
for f in files:
    site, pollutant, year = parse_name(f)
    with open(f"{RAW}/{f}") as fh:
        records = json.load(fh)
    manifest.append({"site_id": site, "pollutant": pollutant, "year": year,
                     "path": f"{RAW}/{f}", "records": len(records), "status": "ok"})

(spark.createDataFrame(manifest)
    .withColumn("run_at", F.current_timestamp())
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.aq_bronze.ingest_manifest"))

spark.sql("""
SELECT count(*)                                     AS total_chunks,
       sum(CASE WHEN records = 0 THEN 1 ELSE 0 END) AS empty_chunks,
       sum(records)                                 AS total_records
FROM workspace.aq_bronze.ingest_manifest
""").show()

+------------+------------+-------------+
|total_chunks|empty_chunks|total_records|
+------------+------------+-------------+
|         576|         120|      3998208|
+------------+------------+-------------+



## 5. Bronze table

Reads every landed file into one Delta table with two audit columns added.
Nothing else is changed - types, names and nesting stay exactly as received.

multiLine is required because each file contains a single JSON array rather than
one record per line.

_metadata.file_path is a built-in Spark column recording the source file of every
row, making any bad row traceable to its origin.

In [0]:
bronze = (spark.read
    .option("multiLine", True)
    .json(f"{RAW}/*.json")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

(bronze.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.aq_bronze.observations"))

spark.table("workspace.aq_bronze.observations").printSchema()

root
 |-- AirQualityCategory: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- DeterminingPollutant: string (nullable = true)
 |-- Hour: long (nullable = true)
 |-- HourDescription: string (nullable = true)
 |-- Parameter: struct (nullable = true)
 |    |-- Category: string (nullable = true)
 |    |-- Frequency: string (nullable = true)
 |    |-- ParameterCode: string (nullable = true)
 |    |-- ParameterDescription: string (nullable = true)
 |    |-- SubCategory: string (nullable = true)
 |    |-- Units: string (nullable = true)
 |    |-- UnitsDescription: string (nullable = true)
 |-- Site_Id: long (nullable = true)
 |-- Value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
spark.sql("""
SELECT
  (SELECT count(*)     FROM workspace.aq_bronze.observations)    AS table_rows,
  (SELECT sum(records) FROM workspace.aq_bronze.ingest_manifest) AS files_on_disk
""").show()

+----------+-------------+
|table_rows|files_on_disk|
+----------+-------------+
|   3998208|      3998208|
+----------+-------------+



In [0]:
spark.sql("""
SELECT
  (SELECT count(*)     FROM workspace.aq_bronze.observations)    AS table_rows,
  (SELECT sum(records) FROM workspace.aq_bronze.ingest_manifest) AS files_on_disk
""").show()

+----------+-------------+
|table_rows|files_on_disk|
+----------+-------------+
|   3998208|      3998208|
+----------+-------------+



In [0]:
spark.sql("""
SELECT Parameter.ParameterCode AS pollutant,
       count(*)                AS total_rows,
       count(Value)            AS has_value,
       count(*) - count(Value) AS nulls,
       round(100.0 * (count(*) - count(Value)) / count(*), 1) AS pct_null,
       min(Date) AS first_date, max(Date) AS last_date
FROM workspace.aq_bronze.observations
GROUP BY 1 ORDER BY 1
""").show()

+---------+----------+---------+------+--------+----------+----------+
|pollutant|total_rows|has_value| nulls|pct_null|first_date| last_date|
+---------+----------+---------+------+--------+----------+----------+
|      NO2|    999552|   858005|141547|    14.2|2020-01-01|2025-12-31|
|    OZONE|    999552|   864833|134719|    13.5|2020-01-01|2025-12-31|
|     PM10|    999552|   900324| 99228|     9.9|2020-01-01|2025-12-31|
|    PM2.5|    999552|   883867|115685|    11.6|2020-01-01|2025-12-31|
+---------+----------+---------+------+--------+----------+----------+



In [0]:
in_data = {r.Site_Id for r in spark.sql(
    "SELECT DISTINCT Site_Id FROM workspace.aq_bronze.observations").collect()}

missing = [s for s in stations if s["Site_Id"] not in in_data]
print(len(in_data), "stations with data;", len(missing), "with none")
for s in missing:
    print(" ", s["Site_Id"], s["SiteName"])

19 stations with data; 5 with none
  70 LINDFIELD
  190 CHULLORA
  574 BARGO
  765 VINEYARD
  1560 MACARTHUR
